# Laboratorio · Sesión 2 — Limpieza y transformación de datos
**PY-EAN · Python para Procesamiento, Calidad y Automatización de Datos** · Universidad Ean

**Caso:** ayer diagnosticaste el archivo de afiliados del fondo de pensiones (ficticio). Hoy tu jefe te pide
**dejarlo listo para análisis**: sin duplicados, con ciudades, géneros y estados uniformes, salarios y fechas con el
tipo correcto, la regional de cada afiliado y un resumen por regional. Al final exportas `afiliados_limpios.csv`.

**Cómo trabajar:**
- Ejecuta cada celda con **Shift + Enter**, de arriba hacia abajo. Si algo se rompe: **Entorno de ejecución → Ejecutar todas**.
- Donde veas `# ✏️ TU CÓDIGO AQUÍ`, escribe tu solución. Las celdas `✔ Verificación` te dicen si vas bien.
- Guarda tu copia: **Archivo → Guardar una copia en Drive**.

> Todos los datos son sintéticos. No corresponden a personas reales.

## Parte 0 · Preparar el entorno y cargar los datos (5 min)

In [ ]:
# ▶️ Ejecuta esta celda primero (Shift + Enter)
import os
import pandas as pd

URL_DATOS = "https://raw.githubusercontent.com/Hackwy402/python-procesamiento-ean/main/datos/"
# Si subiste la carpeta "datos" a Colab (o trabajas en tu PC) se usa la copia local;
# si no, se leen los datos directamente desde GitHub.
for carpeta in ["datos/", "../datos/", "/content/datos/"]:
    if os.path.exists(carpeta + "afiliados.csv"):
        RUTA = carpeta
        break
else:
    RUTA = URL_DATOS

pd.set_option("display.max_columns", 30)
print("Leyendo datos desde:", RUTA)

Leemos documento y teléfono **como texto** (`dtype=str`) para no perder información.
Guardamos el original en `afiliados_raw` y trabajamos sobre una **copia** `df`.

In [ ]:
afiliados_raw = pd.read_csv(RUTA + "afiliados.csv",
                            dtype={"numero_documento": str, "telefono": str})
df = afiliados_raw.copy()

SMMLV = 1_750_905
CORTE = pd.Timestamp("2026-10-31")     # fecha de corte del archivo

print("Filas:", len(df))
df.head()

## Parte 1 · Texto y categorías (15 min)
**✏️ Ejercicio 1.** Limpia `nombres` y `apellidos`: quita espacios al inicio/final y déjalos en **Tipo Título**
(`.str.strip().str.title()`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 2.** Normaliza `genero` para que solo queden `"F"` y `"M"` (los vacíos se quedan vacíos).
Primero `.str.strip().str.upper()`, luego `.map()` con un diccionario.

In [ ]:
mapa_genero = {
    # ✏️ completa el diccionario
}
# ✏️ TU CÓDIGO AQUÍ
df["genero"].value_counts(dropna=False)

**✏️ Ejercicio 3.** Normaliza `estado` para que solo queden `Activo`, `Inactivo` y `Pensionado`.
Pista: `.str.strip().str.capitalize()` y luego `.replace({"Pensionada": "Pensionado"})`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 4 — el caso ciudad.** Hoy hay 39 formas de escribir 15 ciudades.
1. Ejecuta la celda que define `quitar_tildes` y crea la `clave` (minúsculas, sin tildes, sin puntos).
2. Mira la lista de claves distintas y **completa el diccionario** `mapa_ciudad`.
3. Reemplaza la columna `ciudad` con `clave.map(mapa_ciudad)`.

In [ ]:
import unicodedata

def quitar_tildes(texto):
    if pd.isna(texto):
        return texto
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

clave = (df["ciudad"].str.strip().str.lower()
           .map(quitar_tildes)
           .str.replace(".", "", regex=False))
print(len(clave.unique()), "claves distintas:")
print(sorted(clave.unique()))

In [ ]:
mapa_ciudad = {
    "bogota": "Bogotá D.C.",
    "medellin": "Medellín",
    # ✏️ completa con TODAS las claves de la lista de arriba
}
df["ciudad"] = clave.map(mapa_ciudad)
df["ciudad"].value_counts()

In [ ]:
# ✔ Verificación
print("Ciudades distintas:", df["ciudad"].nunique(), "(esperado: 15)")
print("Ciudades sin mapear (NaN):", df["ciudad"].isna().sum(), "(esperado: 0)")

## Parte 2 · Documentos, números y fechas (15 min)
**✏️ Ejercicio 5.** Quita puntos y espacios de `numero_documento` con
`.str.replace(r"[.\s]", "", regex=True)`. ¿Cuántas filas tenían puntos o espacios antes?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 6.** Convierte `salario_base` a número: primero quita los puntos de miles
(`.str.replace(".", "", regex=False)`), luego `pd.to_numeric(..., errors="coerce")`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 7.** Regla de negocio: un salario **≤ 0** no es válido. ¿Cuántos hay? Márcalos como vacíos
(`df.loc[condición, "salario_base"] = None`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 8.** Convierte `fecha_nacimiento` y `fecha_afiliacion` a fechas. Vienen mezcladas en
`aaaa-mm-dd` y `dd/mm/aaaa`: convierte con cada formato (`errors="coerce"`) y combina con `.fillna()`.

In [ ]:
for col in ["fecha_nacimiento", "fecha_afiliacion"]:
    iso = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")
    latino = # ✏️ TU CÓDIGO AQUÍ
    df[col] = # ✏️ TU CÓDIGO AQUÍ
    print(col, "→", df[col].dtype, "| sin convertir:", df[col].isna().sum())

**✏️ Ejercicio 9.** Hay fechas de nacimiento **imposibles** (en el futuro o antes de 1920). ¿Cuántas? Muéstralas y
márcalas como vacías (`pd.NaT`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 10.** Calcula la `edad` y la `antiguedad_anios` (años desde la afiliación) a la fecha de `CORTE`.
Pista: `(CORTE - df["fecha_nacimiento"]).dt.days // 365.25`. ¿Cuál es la edad mínima, promedio y máxima?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

## Parte 3 · Nulos y duplicados (10 min)
**✏️ Ejercicio 11.** Muestra los nulos por columna tras la limpieza. Luego rellena **solo** `genero` con `"Sin dato"`
(es una categoría para reportes). ¿Por qué **no** rellenamos `salario_base` con el promedio?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

✍️ *¿Por qué no imputar el salario?* 

**✏️ Ejercicio 12.** ¿Cuántas filas son **duplicados exactos**? Elimínalas con `drop_duplicates()`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 13.** Ahora busca afiliados con el **mismo documento** (ignorando vacíos). Revisa algunos pares: ¿en
qué se diferencian? Luego conserva solo la primera aparición.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

In [ ]:
# ✔ Verificación
print("Filas:", len(df), "(esperado: 3000)")

## Parte 4 · Columnas derivadas y unión con el catálogo (15 min)
**✏️ Ejercicio 14.** Crea `aporte_esperado` (16 % del salario, redondeado) y `rango_salarial` con `pd.cut`
en los rangos `< 1 SMMLV`, `1-2 SMMLV`, `2-4 SMMLV`, `> 4 SMMLV` (usa `right=False`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 15.** Lee la hoja `ciudades` de `oficinas.xlsx` **dos veces**: sin `dtype` y con
`dtype={"codigo_dane": str}`. Compara el código de Medellín. ¿Qué pasó?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 16.** Une `df` con `cat` (columnas `ciudad`, `codigo_dane`, `regional`) usando `merge` con
`how="left"` e `indicator=True`. Verifica que **todos** encontraron regional y luego elimina la columna `_merge`.

*Bonus:* haz el mismo merge con `afiliados_raw` (ciudad sin limpiar). ¿Cuántos quedan sin regional?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

## Parte 5 · Agregar, resumir y exportar (15 min)
**✏️ Ejercicio 17.** Con `groupby("regional").agg(...)` calcula por regional: número de afiliados, salario promedio,
salario mediano y edad promedio. Ordena de mayor a menor número de afiliados.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 18.** Arma una tabla dinámica (`pivot_table`) de afiliados por **regional × estado**, con totales.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 19.** Exporta `df` a `afiliados_limpios.csv` (sin índice) y muestra un resumen **antes vs. después**.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

> En Colab el archivo queda en el panel **Archivos** (ícono de carpeta, a la izquierda): clic derecho → **Descargar**.
> Guárdalo: lo usaremos en la Sesión 3.

## 🎯 Reto de cierre — PQRS
Carga `pqrs.csv`, convierte `fecha_radicacion` y `fecha_respuesta` a fechas y calcula `dias_respuesta`
(diferencia en días). Luego:
1. Días promedio de respuesta **por tipo** de solicitud.
2. ¿Hay solicitudes con días **negativos**? ¿Qué significa eso?
3. ¿Cuántas están `Cerrado` pero **sin** fecha de respuesta?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ